In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from openfisca_france_indirect_taxation.examples.utils_example import wavg

In [ ]:
output_path = "C:/Users/veve1/OneDrive/Documents/ENSAE PhD/Carbon tax/Output/Figures/Political_feasibility"
output_data_path = 'C:/Users/veve1/OneDrive/Documents/ENSAE PhD/Carbon tax/Output/Data/'

# Neglect of distributive effects

## Vertical effects

In [ ]:
indirect_utility_by_decile =pd.read_csv(os.path.join(output_data_path,'indirect_utility_by_decile.csv'), sep = ',')
indirect_utility_by_decile_node =pd.read_csv(os.path.join(output_data_path,'indirect_utility_by_decile_NoDE.csv'), sep = ',')
to_graph = pd.concat([indirect_utility_by_decile, indirect_utility_by_decile_node], axis = 0)

In [ ]:
# On calcule les valeurs moyennes
Avg_taxes_carburant_total = wavg(indirect_utility_by_decile_node, 'taxes_carburant_total', 'pondmen')
Avg_taxes_carburant_revenu =  Avg_taxes_carburant_total / wavg(indirect_utility_by_decile_node, 'rev_disponible', 'pondmen') * 100
    
Avg_net_transfer_reform = wavg(indirect_utility_by_decile_node, 'Net_transfer_reform', 'pondmen')
Avg_net_transfer_revenu = Avg_net_transfer_reform / wavg(indirect_utility_by_decile_node, 'rev_disponible', 'pondmen') * 100

Avg_total_change_utility = wavg(indirect_utility_by_decile_node, 'total_change_utility', 'pondmen')
Avg_total_change_utility_sur_revenu = Avg_total_change_utility / wavg(indirect_utility_by_decile_node, 'rev_disponible', 'pondmen') * 100

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(data = to_graph, x='niveau_vie_decile', y='total_change_utility_sur_revenu', 
                 hue = 'Neglect_distributed_effect',
                 hue_order = [1, 0.5],
                 palette = sns.color_palette("Paired")[:2],
                 width = 0.9)
plt.axhline(y = Avg_net_transfer_revenu, ls = '--', color = 'orange', linewidth = 2.5)
plt.text(
    x = 2.2, 
    y = -0.13 ,
    s = f'Mean = {Avg_net_transfer_revenu:.2f} %',  # Texte à afficher
    ha = 'right',  # Alignement horizontal
    va = 'bottom',  # Alignement vertical
    fontsize= 18,
    color = 'orange'
)
plt.xlabel('Equivalised income decile', size = 18)
plt.ylabel('Net effect (% of household disp income)', size = 16)
plt.xticks(fontsize = 18)
plt.yticks(np.arange(-.1,0.25,0.05),  fontsize = 16)
handles, labels = ax.get_legend_handles_labels()
labels = ['$\lambda = 1$', '$\lambda = 0.5$']
plt.legend(loc = 'upper right', handles = handles, labels = labels, ncol = 2, title='Discounting distributed effect', fontsize=16, title_fontsize=16)
plt.grid(True, linestyle='--', alpha = 0.7)
plt.savefig(os.path.join(output_path,'Total_change_utility_sur_revenu_node.pdf'), bbox_inches = 'tight')

In [ ]:
share_winners = wavg(indirect_utility_by_decile_node, 'is_winner', 'pondmen') * 100

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(data = to_graph, x='niveau_vie_decile', y=1, color = sns.color_palette("Paired")[5], alpha = 0.8)
ax = sns.barplot(data = to_graph, x='niveau_vie_decile', y='is_winner',
                 hue = 'Neglect_distributed_effect',
                 hue_order = [1, 0.5],
                 palette =[sns.color_palette("Paired")[3]] +  [sns.color_palette("Paired")[3]])
plt.axhline(y = share_winners / 100, ls = '--', color = 'black', linewidth = 2.5)
plt.text(
    x = 4.8, 
    y = 0.3,
    s = f'Total Share of winners = {share_winners:.1f} %',  # Texte à afficher
    ha = 'right',  # Alignement horizontal
    va = 'bottom',  # Alignement vertical
    fontsize= 18,
    color = 'black'
)
plt.xlabel('Equivalised income decile', size = 18)
plt.ylabel('Share of winners/losers', size = 18)
plt.xticks(fontsize = 18)
plt.yticks(  fontsize = 18)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend('',frameon=False)
plt.savefig(os.path.join(output_path,'Share_winners_losers_node.pdf'), bbox_inches = 'tight')

In [ ]:
to_graph.loc[to_graph['niveau_vie_decile'] == 1]

## Horizontal effects

In [ ]:
palette = sns.color_palette("coolwarm",6)[0:2] + sns.color_palette("coolwarm",6)[4:6]

In [ ]:
indirect_utility_by_groups_by_decile = pd.read_csv(os.path.join(output_data_path,'indirect_utility_by_groups_by_decile.csv'), sep = ',', index_col = 0) 
indirect_utility_by_groups_by_decile_node = pd.read_csv(os.path.join(output_data_path,'indirect_utility_by_groups_by_decile_node.csv'), sep = ',', index_col = 0) 
to_graph_by_groups = pd.concat([indirect_utility_by_groups_by_decile, indirect_utility_by_groups_by_decile_node], axis = 0)

In [ ]:
to_graph_by_groups['plot_decile'] = to_graph_by_groups['niveau_vie_decile'] - 0.1 * (to_graph_by_groups['Neglect_distributed_effect'] == 1) + 0.1 * (to_graph_by_groups['Neglect_distributed_effect'] == 0.5)

In [ ]:
to_graph_by_groups.sort_values(by = 'Neglect_distributed_effect', ascending = False, inplace = True)

In [ ]:
plt.figure(figsize=(10, 6))
plt.axhline(y = 0, ls = '--', color = 'black', linewidth = 1)
ax = sns.scatterplot(x = 'plot_decile', 
                y = 'total_change_utility_sur_revenu', 
                hue = 'quartile_depenses_carburants',
                palette = palette,
                s = 150,
                data = to_graph_by_groups,
                markers = ['d', 'o'],
                style = 'Neglect_distributed_effect',
                )
plt.xlabel('Equivalised income decile', size = 18)
plt.ylabel('Net effect (% of household disp income)', size = 16)
plt.xticks(np.arange(1,11,1), fontsize = 18)
plt.yticks(np.arange(-.8,0.7, 0.2),  fontsize = 18)
plt.grid(True, linestyle='--', alpha=0.7)
handles, labels = ax.get_legend_handles_labels()
handles = [
    Line2D([], [], marker='o', linestyle='', color=h.get_color(), markersize=11)
    for h in handles[1:5]
]
ax.legend(loc = 'lower right', title='Fuel expenditure quartile', ncol=4, handles = handles, labels = labels[1:5], fontsize=16, title_fontsize=16)
plt.savefig(os.path.join(output_path,'Total_change_utility_sur_revenu_by_quartile_node.pdf'), bbox_inches = 'tight')

# Social cost of carbon of 100 €/tCO2eq

In [ ]:
output_path = "C:/Users/veve1/OneDrive/Documents/ENSAE PhD/Carbon tax/Output/Figures/Political_feasibility/100_euros_tonne"

## Vertical effects

In [ ]:
indirect_utility_by_decile =pd.read_csv(os.path.join(output_data_path,'indirect_utility_by_decile.csv'), sep = ',')
indirect_utility_by_decile_100_ssc =pd.read_csv(os.path.join(output_data_path,'indirect_utility_by_decile_100_ssc.csv'), sep = ',')
to_graph = pd.concat([indirect_utility_by_decile, indirect_utility_by_decile_100_ssc], axis = 0)

In [ ]:
# On calcule les valeurs moyennes
Avg_taxes_carburant_total = wavg(indirect_utility_by_decile_100_ssc, 'taxes_carburant_total', 'pondmen')
Avg_taxes_carburant_revenu =  Avg_taxes_carburant_total / wavg(indirect_utility_by_decile_100_ssc, 'rev_disponible', 'pondmen') * 100
    
Avg_net_transfer_reform = wavg(indirect_utility_by_decile_100_ssc, 'Net_transfer_reform', 'pondmen')
Avg_net_transfer_revenu = Avg_net_transfer_reform / wavg(indirect_utility_by_decile_100_ssc, 'rev_disponible', 'pondmen') * 100

Avg_total_change_utility = wavg(indirect_utility_by_decile_100_ssc, 'total_change_utility', 'pondmen')
Avg_total_change_utility_sur_revenu = Avg_total_change_utility / wavg(indirect_utility_by_decile_100_ssc, 'rev_disponible', 'pondmen') * 100

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(data = to_graph, x='niveau_vie_decile', y='total_change_utility', 
                 hue = 'social_cost_carbon',
                 palette = sns.color_palette("Paired")[:2])
plt.axhline(y = Avg_total_change_utility, ls = '--', color = 'orange', linewidth = 2.5)
plt.text(
    x = 2.2, 
    y = -8 ,
    s = f'Mean = {Avg_total_change_utility:.1f} €',  # Texte à afficher
    ha = 'right',  # Alignement horizontal
    va = 'bottom',  # Alignement vertical
    fontsize= 18,
    color = 'orange'
)
plt.xlabel('Equivalised income decile', size = 18)
plt.ylabel('Total change in utility (in €)', size = 18)
plt.xticks(fontsize = 18)
plt.yticks(np.arange(-25,30,5),  fontsize = 18)
plt.legend(loc = 'upper right', ncols = 2, title='Social cost of carbon (€/tCO2)', fontsize=16, title_fontsize=16)
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(os.path.join(output_path,'Total_change_utility_100_ssc.pdf'), bbox_inches = 'tight')

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(data = to_graph, x='niveau_vie_decile', y='total_change_utility_sur_revenu', 
                 hue = 'social_cost_carbon',
                 palette = sns.color_palette("Paired")[:2],
                 width = 0.9)
plt.axhline(y = Avg_total_change_utility_sur_revenu, ls = '--', color = 'orange', linewidth = 2.5)
plt.text(
    x = 2.2, 
    y = - 0.03 ,
    s = f'Mean = {Avg_total_change_utility_sur_revenu:.2f} %',  # Texte à afficher
    ha = 'right',  # Alignement horizontal
    va = 'bottom',  # Alignement vertical
    fontsize= 18,
    color = 'orange'
)
plt.xlabel('Equivalised income decile', size = 18)
plt.ylabel('Net effect (% of household disp income)', size = 16)
plt.xticks(fontsize = 18)
plt.yticks(np.arange(-.1,0.25,0.05),  fontsize = 16)
plt.legend(loc = 'upper right', ncols = 2, title='Social cost of carbon (€/tCO2)', fontsize=16, title_fontsize=16)
plt.grid(True, linestyle='--', alpha = 0.7)
plt.savefig(os.path.join(output_path,'Total_change_utility_sur_revenu_100_ssc.pdf'), bbox_inches = 'tight')

In [ ]:
share_winners = wavg(indirect_utility_by_decile_100_ssc, 'is_winner', 'pondmen') * 100

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(data = to_graph, x='niveau_vie_decile', y=1, color = sns.color_palette("Paired")[5], alpha = 0.8)
ax = sns.barplot(data = to_graph, x='niveau_vie_decile', y='is_winner',
                 hue = 'social_cost_carbon',
                 palette =[sns.color_palette("Paired")[3]] +  [sns.color_palette("Paired")[3]])
plt.axhline(y = share_winners / 100, ls = '--', color = 'black', linewidth = 2.5)
plt.text(
    x = 4.8, 
    y = 0.46,
    s = f'Total Share of winners = {share_winners:.1f} %',  # Texte à afficher
    ha = 'right',  # Alignement horizontal
    va = 'bottom',  # Alignement vertical
    fontsize= 18,
    color = 'black'
)
plt.xlabel('Equivalised income decile', size = 18)
plt.ylabel('Share of winners/losers', size = 18)
plt.xticks(fontsize = 18)
plt.yticks(fontsize = 18)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend('',frameon=False)
plt.savefig(os.path.join(output_path,'Share_winners_losers_100_ssc.pdf'), bbox_inches = 'tight')

## Horizontal effects

In [ ]:
palette = sns.color_palette("coolwarm",6)[0:2] + sns.color_palette("coolwarm",6)[4:6]

In [ ]:
indirect_utility_by_groups_by_decile = pd.read_csv(os.path.join(output_data_path,'indirect_utility_by_groups_by_decile.csv'), sep = ',', index_col = 0) 
indirect_utility_by_groups_by_decile_100_ssc = pd.read_csv(os.path.join(output_data_path,'indirect_utility_by_groups_by_decile_100_ssc.csv'), sep = ',', index_col = 0) 
to_graph_by_groups = pd.concat([indirect_utility_by_groups_by_decile, indirect_utility_by_groups_by_decile_100_ssc], axis = 0)

In [ ]:
to_graph_by_groups['plot_decile'] = to_graph_by_groups['niveau_vie_decile'] + 0.1 * (to_graph_by_groups['social_cost_carbon'] == 100) - 0.1 * (to_graph_by_groups['social_cost_carbon'] == 55)

In [ ]:
plt.figure(figsize=(10, 6))
plt.axhline(y = 0, ls = '--', color = 'black', linewidth = 1)
ax = sns.scatterplot(x = 'plot_decile', 
                y = 'total_change_utility_sur_revenu', 
                hue = 'quartile_depenses_carburants',
                palette = palette,
                s = 150,
                data = to_graph_by_groups,
                markers = ['o', 'd'],
                style = 'social_cost_carbon',
                )
plt.xlabel('Equivalised income decile', size = 18)
plt.ylabel('Net effect (% of household disp income)', size = 16)
plt.xticks(np.arange(1,11,1), fontsize = 18)
plt.yticks(np.arange(-.8,0.7, 0.2),  fontsize = 18)
plt.grid(True, linestyle='--', alpha=0.7)
handles, labels = ax.get_legend_handles_labels()
handles = [
    Line2D([], [], marker='o', linestyle='', color=h.get_color(), markersize=11)
    for h in handles[1:5]
]
ax.legend(loc = 'lower right', title='Fuel expenditure quartile', ncol=4, handles = handles, labels = labels[1:5], fontsize=16, title_fontsize=16)
plt.savefig(os.path.join(output_path,'Total_change_utility_sur_revenu_by_quartile_100_ssc.pdf'), bbox_inches = 'tight')